# Logit comparison

In [1]:
import sys
import pandas as pd
import numpy as np
import random
import statsmodels.formula.api as smf
import tensorflow as tf
import plotly.express as px
import plotly.graph_objects as go
import os
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TF_DETERMINISTIC_OPS"] = "1"
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from pathlib import Path

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "src").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.simulation.dgp0 import Tier0Config, simulate_panel
from src.simulation.validation import plot_market_plotly, separation_auc_like
from src.data.feature_eng import feature_eng_syn
from src.model.autoencoder import PriceAutoencoder


In [2]:
#loading baseline data

df = pd.read_parquet("../runs/dgp0/seed_42/baseline/data/features/features_L18.parquet")
df.head()


,market_id,window_start,window_end,window_length,Price 1,Price 2,Price 3,Price 4,Price 5,Price 6,...,CoV_change,zero_change_fraction,AR_1,AR_2,kurtosis_change,max_abs_ret,pos_vol,neg_vol,level_vol,price_range
0,0,0,17,18,0.060889,0.060124,0.055855,0.047379,0.051012,0.050917,...,2.228472,0.117647,-0.077116,0.087097,1.501094,0.029864,0.006910,0.008233,0.029377,0.082866
1,0,1,18,18,0.060124,0.055855,0.047379,0.051012,0.050917,0.055131,...,2.012092,0.058824,-0.182795,0.217825,4.389317,0.057957,0.006910,0.015766,0.038511,0.138185
2,0,2,19,18,0.055855,0.047379,0.051012,0.050917,0.055131,0.036140,...,1.851337,0.058824,-0.020393,0.191116,3.844113,0.057957,0.006910,0.015452,0.046816,0.155376
3,0,3,20,18,0.047379,0.051012,0.050917,0.055131,0.036140,0.055256,...,1.831822,0.058824,-0.019512,0.129449,3.798594,0.057957,0.006910,0.015393,0.053824,0.165478
4,0,4,21,18,0.051012,0.050917,0.055131,0.036140,0.055256,0.060538,...,1.844898,0.058824,-0.049699,0.060755,3.761500,0.057957,0.006829,0.015393,0.058586,0.165478


In [3]:
df = df.loc[df["state_mode"]!= 1]

In [4]:
df.loc[df["state_mode"] == 2, "state_mode"] = 1

In [5]:
df["state_mode"].value_counts()

state_mode
1    13534
0     9572
Name: count, dtype: int64

In [6]:
df.shape

(23106, 39)

## Training Logit

In [7]:
PREDICTORS = [
    "volatility", "zero_change_fraction", "max_abs_ret",
    "AR_1", "price_range"
]
TARGET = "state_mode"

X = df[PREDICTORS].copy()
y = df[TARGET].astype(int)

# Train/validation split keeps class balance for the binary target
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale predictors and keep DataFrame columns for formula-based modeling
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=PREDICTORS, index=X_train.index
)
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val), columns=PREDICTORS, index=X_val.index
)

train_df = X_train_scaled.copy()
train_df[TARGET] = y_train.values
val_df = X_val_scaled.copy()
val_df[TARGET] = y_val.values

In [8]:
formula = TARGET + " ~ " + " + ".join([f'Q("{c}")' for c in PREDICTORS])
model = smf.logit(formula, data=train_df).fit()
model.summary()

Optimization terminated successfully.
         Current function value: 0.395573
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:             state_mode   No. Observations:                18484
Model:                          Logit   Df Residuals:                    18478
Method:                           MLE   Df Model:                            5
Date:                Mon, 09 Mar 2026   Pseudo R-squ.:                  0.4169
Time:                        15:12:16   Log-Likelihood:                -7311.8
converged:                       True   LL-Null:                       -12539.
Covariance Type:            nonrobust   LLR p-value:                     0.000
=============================================================================================
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                     0.1543      0.022      6.879      0.000       0.110       0.198
Q("volatility")              -3.0062      0.100    -29.989      0.000      -3.203      -2.810
Q("zero_change_fraction")     0.0863      0.028      3.129      0.002       0.032       0.140
Q("max_abs_ret")              0.9268      0.084     10.974      0.000       0.761       1.092
Q("AR_1")                    -0.1296      0.024     -5.479      0.000      -0.176      -0.083
Q("price_range")             -0.5903      0.054    -10.961      0.000      -0.696      -0.485
=============================================================================================
"""

In [9]:
y_prob = model.predict(X_val_scaled)
y_pred = (y_prob >= 0.5).astype(int)
y_true = val_df[TARGET]

In [10]:
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)
auc       = roc_auc_score(y_true, y_prob)

print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1 Score : {f1:.3f}")
print(f"ROC-AUC  : {auc:.3f}")

Accuracy : 0.823
Precision: 0.812
Recall   : 0.907
F1 Score : 0.857
ROC-AUC  : 0.880


In [11]:
## loading other datasets to compare

df_beta = pd.read_parquet("../runs/dgp0/seed_42/beta_only/data/features/features_L18.parquet")
df_beta.head()


,market_id,window_start,window_end,window_length,Price 1,Price 2,Price 3,Price 4,Price 5,Price 6,...,CoV_change,zero_change_fraction,AR_1,AR_2,kurtosis_change,max_abs_ret,pos_vol,neg_vol,level_vol,price_range
0,0,0,17,18,0.066714,0.063165,0.056164,0.046739,0.051370,0.051328,...,2.414924,0.058824,-0.127866,0.033985,1.999154,0.035017,0.008844,0.009518,0.031747,0.091082
1,0,1,18,18,0.063165,0.056164,0.046739,0.051370,0.051328,0.055144,...,2.099165,0.058824,-0.210967,0.113727,2.902990,0.057362,0.008844,0.016134,0.039876,0.141498
2,0,2,19,18,0.056164,0.046739,0.051370,0.051328,0.055144,0.032062,...,1.974640,0.058824,-0.069755,0.100084,2.563596,0.057362,0.008844,0.015920,0.047528,0.158129
3,0,3,20,18,0.046739,0.051370,0.051328,0.055144,0.032062,0.056263,...,1.968199,0.058824,-0.067532,0.075652,2.552966,0.057362,0.008844,0.015901,0.054151,0.168061
4,0,4,21,18,0.051370,0.051328,0.055144,0.032062,0.056263,0.062923,...,1.965507,0.058824,-0.095961,0.014479,2.558677,0.057362,0.008861,0.015901,0.058668,0.168061


In [13]:
df_beta = df_beta.loc[df_beta["state_mode"]!= 1]
df_beta.loc[df_beta["state_mode"] == 2, "state_mode"] = 1

In [14]:
X_beta = df_beta[PREDICTORS].copy()
y_beta = df_beta[TARGET].astype(int)

X_scaled_beta = pd.DataFrame(
    scaler.transform(X_beta), columns=PREDICTORS, index=X_beta.index
)

val_beta = X_scaled_beta.copy()
val_beta[TARGET] = y_beta.values

In [16]:
y_prob_beta = model.predict(X_scaled_beta)
y_pred_beta = (y_prob_beta >= 0.5).astype(int)
y_true_beta = val_beta[TARGET]

In [17]:
accuracy_beta  = accuracy_score(y_true_beta, y_pred_beta)
precision_beta = precision_score(y_true_beta, y_pred_beta)
recall_beta    = recall_score(y_true_beta, y_pred_beta)
f1_beta        = f1_score(y_true_beta, y_pred_beta)
auc_beta       = roc_auc_score(y_true_beta, y_pred_beta)

print(f"Accuracy : {accuracy_beta:.3f}")
print(f"Precision: {precision_beta:.3f}")
print(f"Recall   : {recall_beta:.3f}")
print(f"F1 Score : {f1_beta:.3f}")
print(f"ROC-AUC  : {auc_beta:.3f}")

Accuracy : 0.697
Precision: 0.777
Recall   : 0.676
F1 Score : 0.723
ROC-AUC  : 0.701


## Kappa only

In [22]:
df_k = pd.read_parquet("../runs/dgp0/seed_42/kappa_only/data/features/features_L18.parquet")

df_k = df_k.loc[df_k["state_mode"]!= 1]
df_k.loc[df_k["state_mode"] == 2, "state_mode"] = 1

In [23]:
X_k = df_k[PREDICTORS].copy()
y_k = df_k[TARGET].astype(int)

X_scaled_k = pd.DataFrame(
    scaler.transform(X_k), columns=PREDICTORS, index=X_k.index
)

val_k = X_scaled_k.copy()
val_k[TARGET] = y_k.values

In [24]:
y_prob_k = model.predict(X_scaled_k)
y_pred_k = (y_prob_k >= 0.5).astype(int)
y_true_k = val_k[TARGET]

In [25]:
accuracy_k  = accuracy_score(y_true_k, y_pred_k)
precision_k = precision_score(y_true_k, y_pred_k)
recall_k   = recall_score(y_true_k, y_pred_k)
f1_k        = f1_score(y_true_k, y_pred_k)
auc_k      = roc_auc_score(y_true_k, y_pred_k)

print(f"Accuracy : {accuracy_k:.3f}")
print(f"Precision: {precision_k:.3f}")
print(f"Recall   : {recall_k:.3f}")
print(f"F1 Score : {f1_k:.3f}")
print(f"ROC-AUC  : {auc_k:.3f}")

Accuracy : 0.687
Precision: 0.771
Recall   : 0.663
F1 Score : 0.713
ROC-AUC  : 0.693


## Calm fundamentals

In [26]:
df_calm = pd.read_parquet("../runs/dgp0/seed_42/calm_fundamentals/data/features/features_L18.parquet")

df_calm = df_calm.loc[df_calm["state_mode"]!= 1]
df_calm.loc[df_calm["state_mode"] == 2, "state_mode"] = 1

In [27]:
X_calm = df_calm[PREDICTORS].copy()
y_calm = df_calm[TARGET].astype(int)

X_scaled_calm = pd.DataFrame(
    scaler.transform(X_calm), columns=PREDICTORS, index=X_calm.index
)

val_calm = X_scaled_calm.copy()
val_calm[TARGET] = y_calm.values

In [28]:
y_prob_calm = model.predict(X_scaled_calm)
y_pred_calm = (y_prob_calm>= 0.5).astype(int)
y_true_calm = val_calm[TARGET]

In [33]:
prob_comp = y_prob[val_df[TARGET] == 0]
prob_comp

24602    0.397616
20090    0.405810
4547     0.004649
32173    0.156465
32059    0.094879
           ...   
4478     0.156385
17374    0.562532
8295     0.116661
25409    0.000305
566      0.006849
Length: 1915, dtype: float64

In [34]:
tau95 = np.quantile(prob_comp, 0.95)
tau99 = np.quantile(prob_comp, 0.99)

fpr95 = np.mean(y_prob_calm > tau95)
fpr99 = np.mean(y_prob_calm > tau99)

print("tau95:", tau95, "FPR@95:", fpr95)
print("tau99:", tau99, "FPR@99:", fpr99)

tau95: 0.8825442542360536 FPR@95: 0.5065053740046125
tau99: 0.9386104683614787 FPR@99: 0.29611418128018796
